# Agder – 10 m substrate prediction (region-specific)

**Not part of the main national workflow.** This notebook reuses the trained
XGBoost classifier from [02_modelling.ipynb](../02_modelling.ipynb) but runs
predictions at **10 m** resolution for the **Agder** region only, then combines
the model predictions with the authoritative *klassifisering* observations into
a single vector dataset for Agder.

It folds together the relevant parts of the national notebooks:
[03_predict](../03_predict.ipynb) · [04_postprocess](../04_postprocess.ipynb)
· [06_finalise](../06_finalise.ipynb) — scoped to Agder and at 10 m.


## Parameters

In [1]:
from pathlib import Path

import geopandas as gpd
import geoutils as gu
import numpy as np
import pandas as pd
import rasterio as rio
import shapely
import xdem
from osgeo import gdal, ogr
from shapely.strtree import STRtree
from collections import defaultdict

import mnk
import mnk.utils
import mnk.vectorize
import mnk.substrat as subkart

gdal.UseExceptions()

In [2]:
# --- Region-specific configuration ---
REGION = "agder"
FYLKER = ["Agder"]          # restrict all sources to Agder
RES = 10                    # 10 m resolution (national workflow uses 50 m)
nodata = 255
crs = "EPSG:25833"
epsg = int(crs.split(":")[1])

# Output file name stem, tagged with region + resolution so nothing collides
# with the national products.
classifier = subkart.utils.load_classifier()
fname = mnk.utils.to_filename(
    f"nisjedata-substrat-{classifier.__class__.__name__.lower()}",
    REGION, "latest", crs.split(":")[1],
) + f"_{RES}m"

predict_file_unmapped = f"predict_unmapped_{REGION}_{RES}m.tif"
prob_file = f"3band_probability_{REGION}_{RES}m.tif"
print("Output stem:", fname)

Output stem: nisjedata-substrat-xgbclassifier_agder_latest_25833_10m


# 1. Predict (from 03_predict)

Run prediction for Agder only, at 10 m. Wave exposure (bolge) and the DEM50 are
national rasters cropped/reprojected to the Agder bounds; the sea map and depth
points are read for the Agder fylke only.

In [3]:
# Wave exposure (national raster, reprojected to Agder bounds below)
bolge = mnk.sources.bolge_exposure()

# DEM 50 (national)
dem_norge = mnk.sources.dem_data()

print("Loading Agder depth point data...")
gdf_points = mnk.sources.depth_point_data(fylker=FYLKER)
print(f"Depth points: {len(gdf_points):,}")

Loading Agder depth point data...
Depth points: 160,001


In [4]:
print("Loading sea map for Agder...")
gdf_sea_map = mnk.sources.sea_map_basisdata(fylker=FYLKER)
gdf_sea_map = subkart.features.depth_preprocess(gdf_sea_map)
transform, out_shape, bounds = subkart.features.to_raster_shapes(gdf_sea_map, res=RES)
print(f"Output raster shape: {out_shape}  (res={RES} m)")

print("Resampling DEM to Agder grid...")
dem = subkart.utils.resample_dem(dem_norge.crop(bounds), out_shape, transform, crs)

print("Reprojecting wave exposure to Agder grid...")
# Crop bolge to the Agder window *before* reprojecting. geoutils' reproject reads
BOLGE_BUFFER = 1000  # metres
bolge_bbox = gu.Vector(
    gpd.GeoDataFrame(
        geometry=[shapely.box(
            bounds[0] - BOLGE_BUFFER, bounds[1] - BOLGE_BUFFER,
            bounds[2] + BOLGE_BUFFER, bounds[3] + BOLGE_BUFFER,
        )],
        crs=crs,
    )
)
bolge_agder = bolge.crop(bolge_bbox).reproject(
    crs=crs, res=RES,
    bounds=dict(left=bounds[0], bottom=bounds[1], right=bounds[2], top=bounds[3]),
)

print("Building feature arrays...")
X, valid_attrs, out_shape, transform = subkart.features.build(
    dem, gdf_sea_map, bolge_agder, valid_mask=None, res=RES, dtype=np.float32,
    gdf_points=gdf_points,
)
print(f"Feature matrix: {X.shape}")

Loading sea map for Agder...
Output raster shape: (10266, 21222)  (res=10 m)
Resampling DEM to Agder grid...
Reprojecting wave exposure to Agder grid...
Building feature arrays...
Computing interpolated depth raster...
Preparing sea_avg_depth...
Preparing sea_avg_slope...


/home/kim/work/marint-naturkart-nivaR/.pixi/envs/default/lib/python3.13/site-packages/geoutils/raster/raster.py:1822: UserWarning: Setting default nodata -99999 to mask non-finite values found in the array, as no nodata value was defined.
  warnings.warn(


Preparing sea_compactness...


/home/kim/work/marint-naturkart-nivaR/.pixi/envs/default/lib/python3.13/site-packages/geoutils/raster/raster.py:1822: UserWarning: Setting default nodata -99999 to mask non-finite values found in the array, as no nodata value was defined.
  warnings.warn(


Preparing sea_convexity...


/home/kim/work/marint-naturkart-nivaR/.pixi/envs/default/lib/python3.13/site-packages/geoutils/raster/raster.py:1822: UserWarning: Setting default nodata -99999 to mask non-finite values found in the array, as no nodata value was defined.
  warnings.warn(


Preparing wave exposure...
Stacking feature arrays...
Feature matrix: (16123727, 9)


In [5]:
Y_pred = classifier.predict(X)
Y_prob = classifier.predict_proba(X)

# Prediction raster (classes 0=løsbunn, 1=blanding, 2=fastbunn)
pred_map = np.full(out_shape, nodata, dtype=np.uint8)
pred_map[valid_attrs] = Y_pred.astype(np.uint8)
pred_map_masked = np.ma.masked_equal(pred_map, nodata)
pred_raster = gu.Raster.from_array(pred_map_masked, transform=transform, crs=crs, nodata=nodata)
pred_raster.to_file(predict_file_unmapped)
print(f"Saved {predict_file_unmapped}")

# Three-band probability raster (band 1=P(løsbunn), 2=P(blanding), 3=P(fastbunn))
prob_nodata = np.float32(-9999)
prob_bands = np.full((len(classifier.classes_), *out_shape), prob_nodata, dtype=np.float32)
for band_idx in range(len(classifier.classes_)):
    prob_bands[band_idx][valid_attrs] = Y_prob[:, band_idx].astype(np.float32)
with rio.open(
    prob_file, "w", driver="GTiff",
    height=out_shape[0], width=out_shape[1],
    count=len(classifier.classes_), dtype=np.float32,
    crs=crs, transform=transform, nodata=prob_nodata,
    compress="deflate", tiled=True, blockxsize=512, blockysize=512,
) as dst:
    dst.write(prob_bands)
print(f"Saved {prob_file}")

Saved predict_unmapped_agder_10m.tif
Saved 3band_probability_agder_10m.tif


# 2. Postprocess (from 04_postprocess)

Sieve isolated pixels (below `PROB_THRESHOLD`), pad the nodata boundary, then
vectorize both the padded and the processed prediction rasters and assign
`BunnType` / `LM_DK` from the 3-band probabilities.

In [6]:
predict_file = f"{fname}.tif"
PROB_THRESHOLD = 0.60  # filter probability threshold for noise pixels
# Grow the nodata boundary enough to preserve ~50 m of physical padding at 10 m.
PAD_ITER = max(1, round(subkart.features.RESOLUTION / RES))

subkart.utils.sieve_prediction_3class(
    predict_file_unmapped, prob_file, predict_file, nodata=nodata, prob_threshold=PROB_THRESHOLD, res=RES
)
print(f"Saved sieved prediction: {predict_file}")

Saved sieved prediction: nisjedata-substrat-xgbclassifier_agder_latest_25833_10m.tif


In [7]:
# Pad valid pixels by 1 into nodata so vectorized polygons extend slightly beyond
# the data boundary (clean land subtraction at the coast later).
predict_file_padded = predict_file.replace(".tif", "_padded.tif")
prob_file_padded = prob_file.replace(".tif", "_padded.tif")

with rio.open(predict_file) as src:
    arr = src.read(1)
    profile_pred = src.profile.copy()

padded = mnk.vectorize.pad_nodata(arr, nodata=nodata, iterations=PAD_ITER)
n_added = int((padded != nodata).sum() - (arr != nodata).sum())
print(f"Prediction boundary pixels added: {n_added:,}")
del arr

with rio.open(predict_file_padded, "w", **profile_pred) as dst:
    dst.write(padded, 1)
del padded, profile_pred
print(f"Saved padded raster: {predict_file_padded}")

with rio.open(prob_file) as src:
    profile_prob = src.profile.copy()
    prob_nodata = src.nodata
    n_bands = src.count

with rio.open(prob_file_padded, "w", **profile_prob) as dst:
    for b in range(1, n_bands + 1):
        with rio.open(prob_file) as src:
            band_arr = src.read(b)
        band_padded = mnk.vectorize.pad_nodata(band_arr, nodata=prob_nodata, iterations=PAD_ITER)
        if b == 1:
            added = int((band_padded != prob_nodata).sum() - (band_arr != prob_nodata).sum())
            print(f"Probability boundary pixels added (band 1): {added:,}")
        del band_arr
        dst.write(band_padded, b)
        del band_padded
print(f"Saved padded probability raster: {prob_file_padded}")

Prediction boundary pixels added: 1,344,506
Saved padded raster: nisjedata-substrat-xgbclassifier_agder_latest_25833_10m_padded.tif
Probability boundary pixels added (band 1): 1,330,948
Saved padded probability raster: 3band_probability_agder_10m_padded.tif


In [8]:
# Vectorize padded raster (used for clean land subtraction in finalise step).
gdf_padded = mnk.vectorize.vectorize_raster_to_gdf(
    predict_file_padded, field_name="DN", epsg=epsg, nodata=nodata
)
gdf_padded = subkart.utils.assign_bunn_lmdk_from_3band(gdf_padded, prob_file_padded, dn_col="DN")

fname_padded = f"{fname}_padded"
gdf_padded.to_parquet(f"{fname_padded}.geo.parquet", compression="snappy")
print(f"Saved padded vectors: {fname_padded}.geo.parquet ({len(gdf_padded):,} polygons)")
del gdf_padded

Saved padded vectors: nisjedata-substrat-xgbclassifier_agder_latest_25833_10m_padded.geo.parquet (50,794 polygons)


In [9]:
# Vectorize processed (sieved) prediction raster -> region product.
polygons_tmp = f"polygons_processed_{REGION}.gpkg"
mnk.vectorize.with_gdal(predict_file, polygons_tmp, epsg_code=epsg)

gdf = gpd.read_file(polygons_tmp).explode()
gdf = subkart.utils.assign_bunn_lmdk_from_3band(gdf, prob_file, dn_col="DN")

gdf.to_file(f"{fname}.gpkg", driver="GPKG", layer="bunntyper")
gdf.to_parquet(f"{fname}.geo.parquet", compression="snappy")
print(f"Saved prediction vectors: {fname} ({len(gdf):,} polygons)")

Polygons saved to polygons_processed_agder.gpkg
Saved prediction vectors: nisjedata-substrat-xgbclassifier_agder_latest_25833_10m (73,258 polygons)


# 3. Finalise – combine predictions + observations (from 06_finalise)

Merge the authoritative *klassifisering* observations into the Agder model
predictions: observations are clipped to the Agder extent, predictions are erased
beneath them, the result is clipped to the grunnlinje (1 nautical mile) and land
is subtracted. Produces the combined Agder vector dataset.

In [10]:
# LM_DK habitat codes per BunnType (shared source of truth).
LM_DK_MAP = subkart.labelling.LM_DK_MAP

# Agder extent polygon, used to clip the national observations dataset.
agder_extent = shapely.box(*bounds)
agder_extent_gdf = gpd.GeoDataFrame(geometry=[agder_extent], crs=crs)

# Load authoritative classifications (observations) and clip to Agder.
klass = gpd.read_parquet(
    "gs://niva-geodata/MarintNaturKart/results/nisjedata-substrat-klassifisering_norge_latest_25833.geo.parquet",
    columns=["BunnType", "geometry"],
)
klass = gpd.clip(klass, agder_extent_gdf).reset_index(drop=True)
klass = klass[~klass.is_empty].reset_index(drop=True)
print(f"Observations (klassifisering) within Agder: {len(klass):,}")

# Assign LM_DK before remapping blanding so blanding keeps DK_0.
klass["LM_DK"] = klass["BunnType"].map(LM_DK_MAP)
klass["BunnType"] = klass["BunnType"].replace({"blanding": "løsbunn"})
klass["DN"] = klass["BunnType"].map({"løsbunn": 0, "fastbunn": 2})
klass["Sannsynlighet_LM_DK"] = 100.0
klass["Kilde"] = "NGU - Bunnsedimenter (kornstørrelse), detaljert"

# Load Agder model predictions (padded version for clean land subtraction).
pred = gpd.read_parquet(
    f"{fname}_padded.geo.parquet",
    columns=["DN", "BunnType", "LM_DK", "Sannsynlighet_LM_DK", "geometry"],
)
pred["Kilde"] = "NIVA - Substrat Modell"

Observations (klassifisering) within Agder: 0


In [11]:
# Fast erase: bulk spatial join finds all intersecting pairs in one GEOS call.
klass_geoms = klass.geometry.values
pred_geoms = pred.geometry.values.copy()

tree = STRtree(klass_geoms)
pred_hit_idxs, klass_hit_idxs = tree.query(pred_geoms, predicate="intersects")

pred_to_klass = defaultdict(list)
for p, k in zip(pred_hit_idxs.tolist(), klass_hit_idxs.tolist()):
    pred_to_klass[p].append(k)

n_clip = len(pred_to_klass)
print(f"Clipping {n_clip:,} of {len(pred_geoms):,} prediction polygons...")
for i, (pred_i, klass_is) in enumerate(pred_to_klass.items(), 1):
    if i % 1000 == 0:
        print(f"  {i:,} / {n_clip:,}")
    eraser = shapely.unary_union(klass_geoms[klass_is])
    pred_geoms[pred_i] = shapely.difference(pred_geoms[pred_i], eraser)

pred = pred.set_geometry(pred_geoms)
pred_remaining = pred[~pred.geometry.is_empty].explode(index_parts=False)

COLS = ["DN", "BunnType", "LM_DK", "Sannsynlighet_LM_DK", "Kilde", "geometry"]
final = pd.concat([pred_remaining[COLS], klass[COLS]], ignore_index=True)
print(f"Combined predictions + observations: {len(final):,} polygons")

Clipping 0 of 50,794 prediction polygons...
Combined predictions + observations: 50,794 polygons


In [12]:
import os, tempfile

fname_final = mnk.utils.to_filename("nisjedata-substrat", REGION, "2026", crs.split(":")[1]) + f"_{RES}m"

# Clip to grunnlinje (1 nautical mil): remove polygons outside baseline + 1 nm.
print(f"Before grunnlinje clip: {len(final):,} polygons")
with tempfile.TemporaryDirectory() as tmpdir:
    tmp_in = os.path.join(tmpdir, "final_unclipped.gpkg")
    tmp_out = os.path.join(tmpdir, "final_clipped.gpkg")
    final.to_file(tmp_in, driver="GPKG", layer="data")
    grunnlinje_url = "/vsicurl/https://storage.googleapis.com/niva-geodata/MarintNaturKart/aux/grunnlinje_1nautisk.gpkg"
    gdal.VectorTranslate(tmp_out, tmp_in, options=gdal.VectorTranslateOptions(clipSrc=grunnlinje_url))
    final = gpd.read_file(tmp_out).explode(index_parts=False).reset_index(drop=True)

final = final[~final.is_empty].reset_index(drop=True)
print(f"After grunnlinje clip:  {len(final):,} polygons")

# Drop clip artifacts (all-null attribute rows).
attr_cols = ["DN", "BunnType", "LM_DK", "Sannsynlighet_LM_DK", "Kilde"]
null_mask = final[attr_cols].isna().all(axis=1)
print(f"Clip artifacts (all-null rows): {int(null_mask.sum()):,}")
final = final[~null_mask].reset_index(drop=True)

Before grunnlinje clip: 50,794 polygons
After grunnlinje clip:  50,969 polygons
Clip artifacts (all-null rows): 0


In [13]:
# Subtract land.
land = gpd.read_parquet("gs://niva-geodata/MarintNaturKart/aux/Basisdata_Landareal.geo.parquet")
final = mnk.vectorize.subtract_land(final, land)
del land
print(f"After land subtraction: {len(final):,} polygons")

Subtracting land from 2,868 of 50,969 polygons...
After land subtraction: 50,969 polygons


In [14]:
# Save combined Agder vector dataset.
final.to_parquet(f"{fname_final}.geo.parquet", compression="snappy")
final.to_file(f"{fname_final}.gpkg", driver="GPKG", layer="bunntyper")
print(f"Saved combined Agder dataset: {fname_final} ({len(final):,} polygons)")

Saved combined Agder dataset: nisjedata-substrat_agder_2026_25833_10m (50,969 polygons)
